In [64]:
# ============================================================
# CSCE-679 Data Visualization Assignment 1 — Level 1
# Hong Kong Year/Month Temperature Heatmap
# ============================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display


# 1. Load and validate the daily temperature data

csv_path = Path("temperature_daily.csv")

if not csv_path.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path("temperature_daily.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        "Please upload the file named temperature_daily.csv."
    )

df = pd.read_csv(csv_path)
required_columns = {"date", "max_temperature", "min_temperature"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        "CSV must contain: date, max_temperature, min_temperature"
    )

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["max_temperature"] = pd.to_numeric(
    df["max_temperature"], errors="coerce"
)
df["min_temperature"] = pd.to_numeric(
    df["min_temperature"], errors="coerce"
)

df = df.dropna(
    subset=["date", "max_temperature", "min_temperature"]
).copy()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Aggregate daily records by year and month. The monthly maximum is the
# highest daily maximum, and the monthly minimum is the lowest daily minimum.
monthly = (
    df.groupby(["year", "month"], as_index=False)
      .agg(
          max_temperature=("max_temperature", "max"),
          min_temperature=("min_temperature", "min"),
      )
)

# Create a position for every month from 1997 through 2017. Missing months
# keep empty temperature values rather than being treated as zero.
FIRST_YEAR, LAST_YEAR = 1997, 2017
VIEWBOX_WIDTH, VIEWBOX_HEIGHT = 1080, 475
years = list(range(FIRST_YEAR, LAST_YEAR + 1))

lookup = {
    (int(row.year), int(row.month)): {
        "max_temperature": float(row.max_temperature),
        "min_temperature": float(row.min_temperature),
    }
    for row in monthly.itertuples(index=False)
}

matrix_data = []

for year in years:
    for month in range(1, 13):
        record = lookup.get((year, month))
        matrix_data.append({
            "year": year,
            "month": month,
            "max_temperature": (
                record["max_temperature"] if record else None
            ),
            "min_temperature": (
                record["min_temperature"] if record else None
            ),
        })

data_json = json.dumps(matrix_data)

# Print selected monthly summaries to verify the aggregation.
for year, month in [(2002, 3), (2007, 11), (2011, 1), (2017, 8)]:
    record = lookup.get((year, month))
    print(f"{year}-{month:02d}: {record}")


# 2. Create the interactive heatmap

html = r"""
<div id="level1-container" style="
    font-family: Arial, sans-serif;
    color: #111;
    width: 100%;
">
  <h2 style="font-size:24px; margin:0 0 8px 0;">
    Level 1: Year/Month Heatmap
  </h2>
  <h3 style="font-size:20px;font-weight:500;margin:0 0 12px 0;">HK Temperature Heatmap</h3>

  <label for="temperature-mode" style="font-size:18px;">
    Temperature:
  </label>

  <select id="temperature-mode" style="
      font-size:16px;
      padding:4px 6px;
      margin-left:4px;
      margin-bottom:8px;
  ">
    <option value="max_temperature">Maximum temperature</option>
    <option value="min_temperature">Minimum temperature</option>
  </select>

  <div id="heatmap-holder" style="width:100%; overflow-x:auto;">
    <svg id="level1-svg"
         viewBox="0 0 __VIEWBOX_WIDTH__ __VIEWBOX_HEIGHT__"
         style="width:__VIEWBOX_WIDTH__px; max-width:none; height:auto; display:block;">
<g id="heatmap-chart"></g>
    </svg>
  </div>

  <div id="level1-tooltip" style="
      display:none;
      position:fixed;
      z-index:99999;
      background:#fff;
      border:1px solid #888;
      padding:7px 9px;
      border-radius:3px;
      font-size:13px;
      line-height:1.45;
      pointer-events:none;
      box-shadow:0 1px 4px #0002;
  "></div>
</div>

<script src="https://d3js.org/d3.v7.min.js"></script>
<script>
(function () {
  const data = __DATA_JSON__;

  // Keep the chart dimensions and spacing together so the layout is easy to adjust.
  const CONFIG = {
    firstYear: __FIRST_YEAR__,
    lastYear: __LAST_YEAR__,
    cellWidth: 31,
    cellHeight: 31,
    xStart: 60,
    yStart: 28,
    xGap: 9.5,
    yGap: 5.7,
    legendX: 960,
    legendWidth: 18,
    temperatureMin: 0,
    temperatureMax: 40,
    yearFontSize: 9,
    monthFontSize: 10,
    axisOffset: 6,
    tickLength: 4,
    labelOffset: 5,
    axisStrokeWidth: 0.8,
    cellStrokeWidth: 1,
    legendLabelFontSize: 9,
    yearLabelOffset: 9,
    monthLabelBaselineOffset: 3,
    legendLabelOffset: 7,
    tooltipOffset: 12,
    tooltipEdgePadding: 8
  };

  const years = d3.range(CONFIG.firstYear, CONFIG.lastYear + 1);
  const months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
  ];

  const chart = d3.select("#heatmap-chart");
  const tooltip = document.getElementById("level1-tooltip");
  const select = document.getElementById("temperature-mode");

  function positionTooltip(event) {
    const left = Math.min(
      event.clientX + CONFIG.tooltipOffset,
      window.innerWidth - tooltip.offsetWidth - CONFIG.tooltipEdgePadding
    );
    const top = Math.min(
      event.clientY + CONFIG.tooltipOffset,
      window.innerHeight - tooltip.offsetHeight - CONFIG.tooltipEdgePadding
    );
    tooltip.style.left = `${Math.max(CONFIG.tooltipEdgePadding, left)}px`;
    tooltip.style.top = `${Math.max(CONFIG.tooltipEdgePadding, top)}px`;
  }

  // Band scales place each year and month in its own evenly spaced position.
  const xScale = d3.scaleBand()
    .domain(years)
    .range([CONFIG.xStart, CONFIG.xStart + years.length * CONFIG.cellWidth + (years.length - 1) * CONFIG.xGap])
    .paddingInner(CONFIG.xGap / (CONFIG.cellWidth + CONFIG.xGap))
    .paddingOuter(0);
  const yScale = d3.scaleBand()
    .domain(months.map((_, index) => index + 1))
    .range([CONFIG.yStart, CONFIG.yStart + months.length * CONFIG.cellHeight + (months.length - 1) * CONFIG.yGap])
    .paddingInner(CONFIG.yGap / (CONFIG.cellHeight + CONFIG.yGap))
    .paddingOuter(0);

  const cellWidth = xScale.bandwidth();
  const cellHeight = yScale.bandwidth();
  const xPosition = year => xScale(year);
  const yPosition = month => yScale(month);
  const xCenter = year => xScale(year) + xScale.bandwidth() / 2;
  const yCenter = month => yScale(month) + yScale.bandwidth() / 2;

  // Use one fixed temperature scale for both maximum and minimum values.
  // This keeps the same temperature mapped to the same color in both levels.
  const spectralColors = d3.schemeSpectral[11].slice().reverse();
  const temperatureThresholds = [4, 8, 12, 15, 18, 22, 25, 29, 33, 36];
  const temperatureColor = d3.scaleThreshold()
    .domain(temperatureThresholds)
    .range(spectralColors);

  const cells = chart.selectAll(".temperature-cell")
    .data(data, d => `${d.year}-${d.month}`)
    .join("rect")
    .attr("class", "temperature-cell")
    .attr("x", d => xPosition(d.year))
    .attr("y", d => yPosition(d.month))
    .attr("width", cellWidth)
    .attr("height", cellHeight)
    .attr("stroke", "#ffffff")
    .attr("stroke-width", CONFIG.cellStrokeWidth)
    .style("cursor", d =>
      d.max_temperature === null ? "default" : "pointer"
    )
    .on("mouseenter", function(event, d) {
      if (d.max_temperature === null) return;

      tooltip.style.display = "block";
      tooltip.innerHTML =
        `<strong>Date:</strong> ${d.year}-${String(d.month).padStart(2, "0")}; ` +
        `<strong>max:</strong> ${d.max_temperature}°C ` +
        `<strong>min:</strong> ${d.min_temperature}°C`;

      positionTooltip(event);
    })
    .on("mousemove", function(event) {
      positionTooltip(event);
    })
    .on("mouseleave", function() {
      tooltip.style.display = "none";
    });

  // Draw the top and left axes with ticks and labels.
  const gridTop = CONFIG.yStart - CONFIG.axisOffset;
  const gridBottom = yScale(months.length) + yScale.bandwidth();
  const gridLeft = CONFIG.xStart - CONFIG.axisOffset;
  const gridRight = xScale(CONFIG.lastYear) + xScale.bandwidth();

  chart.append("line")
    .attr("x1", CONFIG.xStart).attr("x2", gridRight)
    .attr("y1", gridTop).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-tick")
    .data(years)
    .join("line")
    .attr("class", "year-tick")
    .attr("x1", year => xCenter(year)).attr("x2", year => xCenter(year))
    .attr("y1", gridTop - CONFIG.tickLength).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-label")
    .data(years)
    .join("text")
    .attr("class", "year-label")
    .attr("x", year => xCenter(year))
    .attr("y", gridTop - CONFIG.yearLabelOffset)
    .attr("text-anchor", "middle")
    .attr("font-size", CONFIG.yearFontSize)
    .text(year => year);

  // Draw the left axis line, month tick marks, and month labels.
  chart.append("line")
    .attr("x1", gridLeft).attr("x2", gridLeft)
    .attr("y1", CONFIG.yStart).attr("y2", gridBottom)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-tick")
    .data(months)
    .join("line")
    .attr("class", "month-tick")
    .attr("x1", gridLeft - CONFIG.tickLength).attr("x2", gridLeft)
    .attr("y1", (_, index) => yCenter(index + 1))
    .attr("y2", (_, index) => yCenter(index + 1))
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-label")
    .data(months)
    .join("text")
    .attr("class", "month-label")
    .attr("x", gridLeft - CONFIG.labelOffset)
    .attr("y", (_, index) => yCenter(index + 1) + CONFIG.monthLabelBaselineOffset)
    .attr("text-anchor", "end")
    .attr("font-size", CONFIG.monthFontSize)
    .text(month => month);

  // Draw discrete legend bands using the same colors as the heatmap.
  const legendX = CONFIG.legendX;
  const legendY = CONFIG.yStart;
  const legendWidth = CONFIG.legendWidth;
  const legendHeight = gridBottom - gridTop;
  const bandHeight = legendHeight / spectralColors.length;

  chart.selectAll(".legend-band")
    .data(spectralColors)
    .join("rect")
    .attr("class", "legend-band")
    .attr("x", legendX)
    .attr("y", (_, i) => legendY + i * bandHeight)
    .attr("width", legendWidth)
    .attr("height", bandHeight + 0.2)
    .attr("fill", color => color);

  // Label the endpoints and each temperature threshold used by the color scale.
  const legendLabels = [CONFIG.temperatureMin, ...temperatureThresholds, CONFIG.temperatureMax];
  chart.selectAll(".legend-label")
    .data(legendLabels)
    .join("text")
    .attr("class", "legend-label")
    .attr("x", legendX + legendWidth + CONFIG.legendLabelOffset)
    .attr("y", (_, index) => legendY + index * bandHeight)
    .attr("font-size", CONFIG.legendLabelFontSize)
    .attr("dominant-baseline", "middle")
    .text(value => `${value} Celsius`);

  function updateColors() {
    const metric = select.value;

    cells.attr("fill", d => {
      const value = d[metric];
      if (value === null || !Number.isFinite(value)) {
        return "#eeeeee";
      }

      return temperatureColor(value);
    });
  }

  select.addEventListener("change", updateColors);
  updateColors();
})();
</script>
"""

html = (html.replace("__DATA_JSON__", data_json)
        .replace("__FIRST_YEAR__", str(FIRST_YEAR))
        .replace("__LAST_YEAR__", str(LAST_YEAR))
        .replace("__VIEWBOX_WIDTH__", str(VIEWBOX_WIDTH))
        .replace("__VIEWBOX_HEIGHT__", str(VIEWBOX_HEIGHT)))
display(HTML(html))

2002-03: {'max_temperature': 28.0, 'min_temperature': 14.0}
2007-11: {'max_temperature': 29.0, 'min_temperature': 12.0}
2011-01: {'max_temperature': 18.0, 'min_temperature': 6.0}
2017-08: {'max_temperature': 37.0, 'min_temperature': 25.0}


In [65]:

import json
import pandas as pd
from IPython.display import HTML, display

# --------------------------------------------------
# 1. Load and prepare the daily temperature data
# --------------------------------------------------

df_level2 = pd.read_csv("temperature_daily.csv")

df_level2["date"] = pd.to_datetime(df_level2["date"], errors="coerce")
df_level2["max_temperature"] = pd.to_numeric(
    df_level2["max_temperature"], errors="coerce"
)
df_level2["min_temperature"] = pd.to_numeric(
    df_level2["min_temperature"], errors="coerce"
)

df_level2 = df_level2.dropna(
    subset=["date", "max_temperature", "min_temperature"]
).copy()

df_level2["year"] = df_level2["date"].dt.year
df_level2["month"] = df_level2["date"].dt.month
df_level2["day"] = df_level2["date"].dt.day

# Level 2 covers 2008 through 2017
df_level2 = df_level2[
    (df_level2["year"] >= 2008) &
    (df_level2["year"] <= 2017)
].copy()

# --------------------------------------------------
# 2. Calculate monthly maximum and minimum values
# --------------------------------------------------

monthly = (
    df_level2.groupby(["year", "month"])
    .agg(
        max_temperature=("max_temperature", "max"),
        min_temperature=("min_temperature", "min")
    )
    .reset_index()
)

monthly_records = monthly.to_dict(orient="records")

daily_records = df_level2[
    ["date", "year", "month", "day",
     "max_temperature", "min_temperature"]
].copy()

daily_records["date"] = daily_records["date"].dt.strftime("%Y-%m-%d")
daily_records = daily_records.to_dict(orient="records")

monthly_json = json.dumps(monthly_records)
daily_json = json.dumps(daily_records)

# --------------------------------------------------
# 3. Build the interactive D3.js visualization
# --------------------------------------------------

html_template = r"""
<div id="level2-container">
  <h3>Level 2: Daily Temperature Trends, 2008–2017</h3>

  <p class="l2-description">
    Each cell represents a month. The background shows the monthly
    extreme selected below. Green and cyan lines show daily maximum
    and minimum temperatures.
  </p>

  <div class="l2-controls">
    <span class="l2-label">Monthly background:</span>
    <button id="l2-max" class="l2-button active">
      Maximum Temperature
    </button>
    <button id="l2-min" class="l2-button">
      Minimum Temperature
    </button>
  </div>

  <div id="l2-legend"></div>
  <div id="l2-chart"></div>
  <div id="l2-tooltip"></div>
</div>

<style>
  #level2-container {
    font-family: Arial, sans-serif;
    color: #222;
    max-width: 1200px;
    margin: 15px auto;
  }

  #level2-container h3 {
    margin-bottom: 8px;
  }

  .l2-description {
    line-height: 1.5;
    color: #555;
  }

  .l2-controls {
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    gap: 8px;
    margin: 15px 0;
  }

  .l2-label {
    font-weight: bold;
  }

  .l2-button {
    padding: 8px 12px;
    border: 1px solid #999;
    border-radius: 5px;
    background: white;
    cursor: pointer;
    font-size: 13px;
  }

  .l2-button.active {
    background: #333;
    color: white;
    border-color: #333;
  }

  #l2-chart {
    width: 100%;
    overflow-x: auto;
  }

  #l2-chart svg {
    display: block;
    max-width: 100%;
    height: auto;
  }

  #l2-tooltip {
    position: fixed;
    display: none;
    pointer-events: none;
    background: rgba(25, 25, 25, 0.94);
    color: white;
    padding: 9px 12px;
    border-radius: 5px;
    font-size: 12px;
    line-height: 1.6;
    z-index: 99999;
  }

  #l2-legend {
    margin: 12px 0;
    font-size: 12px;
  }

  .l2-legend-bar {
    display: flex;
    width: 100%;
    max-width: 420px;
    height: 14px;
    margin: 6px 0;
  }

  .l2-legend-bar span {
    flex: 1;
  }

  .l2-legend-labels {
    display: flex;
    justify-content: space-between;
    max-width: 420px;
  }
</style>

<div id="l2-d3-root"></div>

<script>
(function () {
  const monthlyData = __MONTHLY_DATA__;
  const dailyData = __DAILY_DATA__;

  const years = d3.range(2008, 2018);
  const monthNames = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
  ];

  const cellWidth = 78;
  const cellHeight = 58;
  const gapX = 12;
  const gapY = 10;

  const margin = {
    top: 38,
    right: 20,
    bottom: 25,
    left: 48
  };

  const chartWidth =
    margin.left + margin.right +
    years.length * (cellWidth + gapX) - gapX;

  const chartHeight =
    margin.top + margin.bottom +
    12 * (cellHeight + gapY) - gapY;

  // Shared temperature palette and thresholds
  const thresholds = [4, 8, 12, 15, 18, 22, 25, 29, 33, 36];

  const palette = d3.schemeSpectral[11].slice().reverse();

  const colorScale = d3.scaleThreshold()
    .domain(thresholds)
    .range(palette);

  // Scales for the daily lines inside each monthly cell
  const xScale = d3.scaleLinear()
    .domain([1, 31])
    .range([5, cellWidth - 5]);

  const yScale = d3.scaleLinear()
    .domain([0, 40])
    .range([cellHeight - 6, 6]);

  const root = d3.select("#l2-d3-root");

  root.selectAll("*").remove();

  const svg = root.append("svg")
    .attr("viewBox", `0 0 ${chartWidth} ${chartHeight}`)
    .attr("width", chartWidth)
    .attr("height", chartHeight)
    .attr("role", "img")
    .attr(
      "aria-label",
      "Monthly temperature heatmap with daily maximum and minimum lines"
    );

  // Year headings
  svg.selectAll(".l2-year")
    .data(years)
    .enter()
    .append("text")
    .attr("class", "l2-year")
    .attr("x", (d, i) =>
      margin.left + i * (cellWidth + gapX) + cellWidth / 2
    )
    .attr("y", 22)
    .attr("text-anchor", "middle")
    .attr("font-size", 12)
    .attr("font-weight", "bold")
    .text(d => d);

  // Month labels
  svg.selectAll(".l2-month")
    .data(monthNames)
    .enter()
    .append("text")
    .attr("class", "l2-month")
    .attr("x", margin.left - 9)
    .attr("y", (d, i) =>
      margin.top + i * (cellHeight + gapY) + cellHeight / 2 + 4
    )
    .attr("text-anchor", "end")
    .attr("font-size", 11)
    .text(d => d);

  // Index data by year-month for quick lookup
  const monthlyMap = new Map(
    monthlyData.map(d => [`${d.year}-${d.month}`, d])
  );

  const dailyMap = d3.group(
    dailyData,
    d => `${d.year}-${d.month}`
  );

  // Build a complete grid so missing months remain blank
  const cells = [];

  years.forEach(year => {
    d3.range(1, 13).forEach(month => {
      cells.push({
        year,
        month,
        monthly: monthlyMap.get(`${year}-${month}`) || null,
        daily: dailyMap.get(`${year}-${month}`) || []
      });
    });
  });

  const tooltip = d3.select("#l2-tooltip");

  function showTooltip(event, html) {
    tooltip
      .style("display", "block")
      .html(html)
      .style("left", (event.clientX + 12) + "px")
      .style("top", (event.clientY + 12) + "px");
  }

  function hideTooltip() {
    tooltip.style("display", "none");
  }

  const cellGroups = svg.selectAll(".l2-cell")
    .data(cells)
    .enter()
    .append("g")
    .attr("class", "l2-cell")
    .attr("transform", d => {
      const col = years.indexOf(d.year);
      const row = d.month - 1;

      const x = margin.left + col * (cellWidth + gapX);
      const y = margin.top + row * (cellHeight + gapY);

      return `translate(${x},${y})`;
    });

  // Monthly background rectangles
  cellGroups.append("rect")
    .attr("class", "l2-background")
    .attr("width", cellWidth)
    .attr("height", cellHeight)
    .attr("rx", 2)
    .attr("stroke", "#ffffff")
    .attr("stroke-width", 1)
    .attr("fill", d =>
      d.monthly ? colorScale(d.monthly.max_temperature) : "#eeeeee"
    )
    .on("mousemove", function(event, d) {
      if (!d.monthly) {
        hideTooltip();
        return;
      }

      const field = currentMode === "max"
        ? "max_temperature"
        : "min_temperature";

      const label = currentMode === "max"
        ? "Monthly maximum"
        : "Monthly minimum";

      showTooltip(
        event,
        `<strong>${monthNames[d.month - 1]} ${d.year}</strong><br>` +
        `${label}: ${d.monthly[field].toFixed(1)} °C`
      );
    })
    .on("mouseleave", hideTooltip);

  // Daily line generators
  const lineMax = d3.line()
    .defined(d => Number.isFinite(+d.max_temperature))
    .x(d => xScale(+d.day))
    .y(d => yScale(+d.max_temperature));

  const lineMin = d3.line()
    .defined(d => Number.isFinite(+d.min_temperature))
    .x(d => xScale(+d.day))
    .y(d => yScale(+d.min_temperature));

  // Both daily lines remain visible in either background mode
  cellGroups.each(function(cell) {
    if (!cell.daily.length) return;

    const group = d3.select(this);

    const orderedDays = cell.daily.slice()
      .sort((a, b) => +a.day - +b.day);

    group.append("path")
      .datum(orderedDays)
      .attr("class", "l2-daily-max")
      .attr("d", lineMax)
      .attr("fill", "none")
      .attr("stroke", "#00a86b")
      .attr("stroke-width", 1.5)
      .attr("stroke-linejoin", "round")
      .attr("stroke-linecap", "round")
      .attr("pointer-events", "none");

    group.append("path")
      .datum(orderedDays)
      .attr("class", "l2-daily-min")
      .attr("d", lineMin)
      .attr("fill", "none")
      .attr("stroke", "#39bde0")
      .attr("stroke-width", 1.5)
      .attr("stroke-linejoin", "round")
      .attr("stroke-linecap", "round")
      .attr("pointer-events", "none");

    // Invisible hit targets provide individual-day tooltips
    const dailyPoints = orderedDays.flatMap(day => [
      {
        ...day,
        type: "Maximum",
        value: +day.max_temperature
      },
      {
        ...day,
        type: "Minimum",
        value: +day.min_temperature
      }
    ]);

    group.selectAll(".l2-daily-point")
      .data(dailyPoints)
      .enter()
      .append("circle")
      .attr("class", "l2-daily-point")
      .attr("cx", d => xScale(+d.day))
      .attr("cy", d => yScale(d.value))
      .attr("r", 5)
      .attr("fill", "transparent")
      .attr("stroke", "none")
      .style("cursor", "crosshair")
      .on("mousemove", function(event, d) {
        showTooltip(
          event,
          `<strong>${d.date}</strong><br>` +
          `${d.type} temperature: ${d.value.toFixed(1)} °C`
        );
      })
      .on("mouseleave", hideTooltip);
  });

  // Legend
  const legend = d3.select("#l2-legend");

  legend.html(
    `<strong>Temperature scale (°C)</strong>` +
    `<div class="l2-legend-bar"></div>` +
    `<div class="l2-legend-labels"><span>0 °C</span><span>40 °C</span></div>` +
    `<p style="margin-top:10px">` +
    `<span style="color:#00a86b">━</span> Daily maximum &nbsp;&nbsp;` +
    `<span style="color:#39bde0">━</span> Daily minimum</p>`
  );

  legend.select(".l2-legend-bar")
    .selectAll("span")
    .data(d3.range(11))
    .enter()
    .append("span")
    .style("background", d => palette[d]);

  // Toggle between monthly maximum and minimum backgrounds
  let currentMode = "max";

  function updateLevel2(mode) {
    currentMode = mode;

    cellGroups.select(".l2-background")
      .attr("fill", d => {
        if (!d.monthly) return "#eeeeee";

        const value = mode === "max"
          ? d.monthly.max_temperature
          : d.monthly.min_temperature;

        return colorScale(value);
      });

    d3.select("#l2-max")
      .classed("active", mode === "max")
      .attr("aria-pressed", mode === "max");

    d3.select("#l2-min")
      .classed("active", mode === "min")
      .attr("aria-pressed", mode === "min");

    hideTooltip();
  }

  d3.select("#l2-max").on("click", () => updateLevel2("max"));
  d3.select("#l2-min").on("click", () => updateLevel2("min"));

  updateLevel2("max");
})();
</script>
"""

# Replace placeholders with prepared JSON data
html = (
    html_template
    .replace("__MONTHLY_DATA__", monthly_json)
    .replace("__DAILY_DATA__", daily_json)
)

# Load D3.js, then render the visualization
display(HTML(
    """
    <script src="https://d3js.org/d3.v7.min.js"></script>
    """
    + html
))
